In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/goddumuriraghu/json-summary/adst_v2_seed2024_history.csv
/kaggle/input/datasets/goddumuriraghu/json-summary/ft_flat_matched_seed42_history.csv
/kaggle/input/datasets/goddumuriraghu/json-summary/adst_v2_multiseed_summary.json
/kaggle/input/datasets/goddumuriraghu/json-summary/adst_v2_seed42_history.csv
/kaggle/input/datasets/goddumuriraghu/json-summary/adst_v2_seed123_history.csv
/kaggle/input/datasets/goddumuriraghu/json-summary/ft_flat_matched_seed2024_history.csv
/kaggle/input/datasets/goddumuriraghu/json-summary/ft_flat_matched_seed123_history.csv
/kaggle/input/datasets/goddumuriraghu/json-summary/ft_flat_matched_multiseed_summary.json


# Cell 1 — verify all required files are present

In [2]:
# Cell 1 — verify all required files are present
import os

input_path = "/kaggle/input/datasets/goddumuriraghu/json-summary/"

required = [
    "ft_flat_matched_seed42_history.csv",
    "ft_flat_matched_seed123_history.csv",
    "ft_flat_matched_seed2024_history.csv",
    "ft_flat_matched_multiseed_summary.json",
    "adst_v2_seed42_history.csv",
    "adst_v2_seed123_history.csv",
    "adst_v2_seed2024_history.csv",
    "adst_v2_multiseed_summary.json",
]

print("Checking required files:")
all_found = True
for f in required:
    path = os.path.join(input_path, f)
    found = os.path.exists(path)
    print(f"  {'✓' if found else '✗'}  {f}")
    if not found:
        all_found = False

print()
print("All files found!" if all_found else "MISSING FILES — check dataset name above")

Checking required files:
  ✓  ft_flat_matched_seed42_history.csv
  ✓  ft_flat_matched_seed123_history.csv
  ✓  ft_flat_matched_seed2024_history.csv
  ✓  ft_flat_matched_multiseed_summary.json
  ✓  adst_v2_seed42_history.csv
  ✓  adst_v2_seed123_history.csv
  ✓  adst_v2_seed2024_history.csv
  ✓  adst_v2_multiseed_summary.json

All files found!


# Cell 2 — copy files to working dir and run plot script

In [3]:
# Cell 2 — copy files to working dir and run plot script
import shutil, os

input_path  = "/kaggle/input/datasets/goddumuriraghu/json-summary/"
working_dir = "/kaggle/working"

# Copy all training output files to working directory
for f in os.listdir(input_path):
    if f.endswith(".csv") or f.endswith(".json"):
        shutil.copy(
            os.path.join(input_path, f),
            os.path.join(working_dir, f)
        )
        print(f"  Copied: {f}")

print("\nAll files copied to working directory.")

  Copied: adst_v2_seed2024_history.csv
  Copied: ft_flat_matched_seed42_history.csv
  Copied: adst_v2_multiseed_summary.json
  Copied: adst_v2_seed42_history.csv
  Copied: adst_v2_seed123_history.csv
  Copied: ft_flat_matched_seed2024_history.csv
  Copied: ft_flat_matched_seed123_history.csv
  Copied: ft_flat_matched_multiseed_summary.json

All files copied to working directory.


# Cell 3 — run the plot comparison script

In [4]:
"""
==========================================================
PLOT_COMPARISON.PY  v2  — Focused Model Comparison Graphs
==========================================================
Run AFTER both training scripts complete.

Produces 11 focused graphs:
  ADST vs FT-Transformer head-to-head:
    01  Val accuracy curves (mean ± std)
    02  Training loss curves
    03  Convergence speed at key epochs
    04  Training time deep-dive
    05  Seed stability boxplot

  ADST vs FT vs XGBoost:
    06  Final test accuracy bar chart
    07  Per-class recall heatmap (24 classes)
    08  Per-class delta: where ADST beats FT (key insight)

  ADST unique advantages:
    09  Semantic group attention weights
    10  Accuracy vs Training Time scatter (efficiency frontier)

  Paper figure:
    11  3×2 combined publication figure

USAGE:
  python plot_comparison.py
==========================================================
"""

import os, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from matplotlib.colors import LinearSegmentedColormap

# ── Configuration ─────────────────────────────────────────
FT_PREFIX   = "ft_flat_matched"
ADST_PREFIX = "adst_v2"
SEEDS       = [42, 123, 2024]
OUTPUT_DIR  = "comparison_plots"

# Known results from your completed runs
XGBOOST_ACC      = 0.9006
XGBOOST_F1       = 0.9027
FT_EPOCH_TIME_S  = 109     # seconds per epoch (from training logs)
ADST_EPOCH_TIME_S = 12     # seconds per epoch (from training logs)
N_EPOCHS         = 50
N_SEEDS          = 3

# Per-class XGBoost recalls (alphabetical order, matches class_names)
XGB_RECALLS = {
    "Benign_Final":             0.903,
    "DDoS-ACK_Fragmentation":  1.000,
    "DDoS-HTTP_Flood":         1.000,
    "DDoS-ICMP_Flood":         0.703,
    "DDoS-ICMP_Fragmentation": 1.000,
    "DDoS-PSHACK_Flood":       0.996,
    "DDoS-RSTFINFlood":        0.995,
    "DDoS-SYN_Flood":          0.993,
    "DDoS-SlowLoris":          1.000,
    "DDoS-SynonymousIP_Flood": 0.974,
    "DDoS-TCP_Flood":          0.984,
    "DDoS-UDP_Flood":          0.797,
    "DDoS-UDP_Fragmentation":  1.000,
    "DoS-HTTP_Flood":          0.901,
    "DoS-SYN_Flood":           0.953,
    "DoS-TCP_Flood":           0.971,
    "DoS-UDP_Flood":           0.908,
    "Mirai-greeth_flood":      0.632,
    "Mirai-greip_flood":       0.681,
    "Mirai-udpplain":          0.636,
    "Recon-HostDiscovery":     0.934,
    "Recon-OSScan":            0.811,
    "Recon-PortScan":          0.843,
    "VulnerabilityScan":       1.000,
}

# Colors
C_ADST  = "#2563EB"   # blue  — ADST
C_FT    = "#DC2626"   # red   — FT-Transformer
C_XGB   = "#16A34A"   # green — XGBoost
C_WIN   = "#059669"   # emerald — ADST wins
C_LOSE  = "#EF4444"   # red   — ADST loses

os.makedirs(OUTPUT_DIR, exist_ok=True)
plt.rcParams.update({
    "font.family":     "DejaVu Sans",
    "font.size":       11,
    "axes.titlesize":  13,
    "axes.titleweight":"bold",
    "axes.labelsize":  11,
    "legend.fontsize": 10,
    "figure.dpi":      150,
    "axes.spines.top":    False,
    "axes.spines.right":  False,
})

# ── Data loading helpers ───────────────────────────────────

def load_histories(prefix, seeds):
    out = []
    for s in seeds:
        p = f"{prefix}_seed{s}_history.csv"
        if os.path.exists(p):
            out.append(pd.read_csv(p))
        else:
            print(f"  WARNING: {p} not found")
    return out

def load_summary(prefix):
    p = f"{prefix}_multiseed_summary.json"
    if os.path.exists(p):
        with open(p) as f:
            return json.load(f)
    print(f"  WARNING: {p} not found")
    return None

def mean_std(histories, col):
    n = min(len(h) for h in histories)
    v = np.array([h[col].values[:n] for h in histories])
    return np.arange(1, n+1), v.mean(0), v.std(0)

def save(fig, name):
    p = os.path.join(OUTPUT_DIR, f"{name}.png")
    fig.savefig(p, dpi=300, bbox_inches="tight", facecolor="white")
    print(f"  Saved → {p}")
    plt.close(fig)

# ── Load everything ────────────────────────────────────────
print("Loading data...")
ft_hist    = load_histories(FT_PREFIX,   SEEDS)
adst_hist  = load_histories(ADST_PREFIX, SEEDS)
ft_sum     = load_summary(FT_PREFIX)
adst_sum   = load_summary(ADST_PREFIX)

ft_test_accs   = [r["test_acc"]   for r in ft_sum]   if ft_sum   else []
adst_test_accs = [r["test_acc"]   for r in adst_sum] if adst_sum else []
ft_mean   = np.mean(ft_test_accs)   if ft_test_accs   else 0
adst_mean = np.mean(adst_test_accs) if adst_test_accs else 0

print(f"  FT seeds loaded:   {len(ft_hist)}")
print(f"  ADST seeds loaded: {len(adst_hist)}")


# ══════════════════════════════════════════════════════════
# PLOT 01 — Validation Accuracy Curves
# ══════════════════════════════════════════════════════════
print("\nPlot 01: Val accuracy curves...")

fig, ax = plt.subplots(figsize=(10, 6))
e_ft, m_ft, s_ft = mean_std(ft_hist,   "val_acc")
e_ad, m_ad, s_ad = mean_std(adst_hist, "val_acc")

ax.plot(e_ft, m_ft*100, color=C_FT,   lw=2.5,
        label=f"Flat FT-Transformer  (mean={ft_mean*100:.2f}%)")
ax.fill_between(e_ft, (m_ft-s_ft)*100, (m_ft+s_ft)*100,
                color=C_FT, alpha=0.12, label="_nolegend_")

ax.plot(e_ad, m_ad*100, color=C_ADST, lw=2.5,
        label=f"ADST — Semantic Tokens  (mean={adst_mean*100:.2f}%)")
ax.fill_between(e_ad, (m_ad-s_ad)*100, (m_ad+s_ad)*100,
                color=C_ADST, alpha=0.12, label="_nolegend_")

ax.axhline(XGBOOST_ACC*100, color=C_XGB, lw=1.5, ls="--",
           alpha=0.8, label=f"XGBoost ceiling  ({XGBOOST_ACC*100:.2f}%)")

# Annotate final values
ax.annotate(f"{m_ft[-1]*100:.2f}%", xy=(e_ft[-1], m_ft[-1]*100),
            xytext=(8, 0), textcoords="offset points",
            color=C_FT, fontweight="bold", va="center")
ax.annotate(f"{m_ad[-1]*100:.2f}%", xy=(e_ad[-1], m_ad[-1]*100),
            xytext=(8, 0), textcoords="offset points",
            color=C_ADST, fontweight="bold", va="center")

ax.set_xlabel("Epoch")
ax.set_ylabel("Validation Accuracy (%)")
ax.set_title("Validation Accuracy: ADST vs Flat FT-Transformer\n"
             "Shaded region = ±1 std across 3 seeds  |  Higher is better")
ax.legend(loc="lower right")
ax.set_xlim(1, max(e_ft[-1], e_ad[-1]) + 3)
ax.set_ylim(75, 92)
ax.grid(True, alpha=0.25)
save(fig, "01_val_accuracy_curve")


# ══════════════════════════════════════════════════════════
# PLOT 02 — Training Loss Curves
# ══════════════════════════════════════════════════════════
print("Plot 02: Training loss curves...")

fig, ax = plt.subplots(figsize=(10, 6))
e_ft, m_ft, s_ft = mean_std(ft_hist,   "train_loss")
e_ad, m_ad, s_ad = mean_std(adst_hist, "train_loss")

ax.plot(e_ft, m_ft, color=C_FT,   lw=2.5, label="Flat FT-Transformer")
ax.fill_between(e_ft, m_ft-s_ft, m_ft+s_ft, color=C_FT, alpha=0.12)
ax.plot(e_ad, m_ad, color=C_ADST, lw=2.5, label="ADST — Semantic Tokens")
ax.fill_between(e_ad, m_ad-s_ad, m_ad+s_ad, color=C_ADST, alpha=0.12)

ax.set_xlabel("Epoch")
ax.set_ylabel("Training Loss (Cross-Entropy)")
ax.set_title("Training Loss: ADST vs Flat FT-Transformer\n"
             "Shaded region = ±1 std across 3 seeds  |  Lower is better")
ax.legend()
ax.grid(True, alpha=0.25)
save(fig, "02_training_loss_curve")


# ══════════════════════════════════════════════════════════
# PLOT 03 — Convergence Speed at Key Epochs
# ══════════════════════════════════════════════════════════
print("Plot 03: Convergence speed...")

key_epochs = [1, 3, 5, 10, 15, 20, 30, 40, 50]

def acc_at(histories, epoch):
    vals = [h["val_acc"].values[epoch-1]*100
            for h in histories if len(h) >= epoch]
    return (np.mean(vals), np.std(vals)) if vals else (None, None)

ft_pts   = [(e, *acc_at(ft_hist,   e)) for e in key_epochs]
adst_pts = [(e, *acc_at(adst_hist, e)) for e in key_epochs]
ft_pts   = [(e, m, s) for e, m, s in ft_pts   if m is not None]
adst_pts = [(e, m, s) for e, m, s in adst_pts if m is not None]

fig, ax = plt.subplots(figsize=(10, 6))

e_ft   = [x[0] for x in ft_pts];   m_ft_c = [x[1] for x in ft_pts];   s_ft_c = [x[2] for x in ft_pts]
e_ad   = [x[0] for x in adst_pts]; m_ad_c = [x[1] for x in adst_pts]; s_ad_c = [x[2] for x in adst_pts]

ax.errorbar(e_ft, m_ft_c, yerr=s_ft_c,
            fmt="o-", color=C_FT,   lw=2.5, ms=8, capsize=5,
            label="Flat FT-Transformer")
ax.errorbar(e_ad, m_ad_c, yerr=s_ad_c,
            fmt="s-", color=C_ADST, lw=2.5, ms=8, capsize=5,
            label="ADST — Semantic Tokens")
ax.axhline(XGBOOST_ACC*100, color=C_XGB, lw=1.5, ls="--",
           alpha=0.8, label=f"XGBoost ({XGBOOST_ACC*100:.2f}%)")

# Wall-clock time axis on top
ax2 = ax.twiny()
ax2.set_xlim(ax.get_xlim())
tick_epochs = [1, 10, 20, 30, 40, 50]
ax2.set_xticks(tick_epochs)
ax2.set_xticklabels([f"{e*ADST_EPOCH_TIME_S//60}m\n(ADST)" for e in tick_epochs],
                     fontsize=8, color=C_ADST)

ax.set_xlabel("Epoch")
ax.set_ylabel("Validation Accuracy (%)")
ax.set_title("Convergence Speed: Who Learns Faster?\n"
             "Error bars = ±1 std across 3 seeds")
ax.legend(loc="lower right")
ax.set_xticks(key_epochs)
ax.grid(True, alpha=0.25)
ax.set_ylim(74, 92)
save(fig, "03_convergence_speed")


# ══════════════════════════════════════════════════════════
# PLOT 04 — Training Time Deep-Dive
# ══════════════════════════════════════════════════════════
print("Plot 04: Training time comparison...")

fig, axes = plt.subplots(1, 3, figsize=(15, 6))

# Panel A: Epoch time bar
ax = axes[0]
bars = ax.bar(["FT-Transformer", "ADST"],
              [FT_EPOCH_TIME_S, ADST_EPOCH_TIME_S],
              color=[C_FT, C_ADST], width=0.5)
ax.bar_label(bars, labels=[f"{FT_EPOCH_TIME_S}s", f"{ADST_EPOCH_TIME_S}s"],
             padding=3, fontweight="bold")
ax.set_ylabel("Seconds per Epoch")
ax.set_title("(a) Time per Epoch\n(lower is better)")
ax.set_ylim(0, 130)
speedup = FT_EPOCH_TIME_S / ADST_EPOCH_TIME_S
ax.text(0.5, 0.88, f"ADST is {speedup:.0f}× faster",
        transform=ax.transAxes, ha="center", fontsize=12,
        color=C_ADST, fontweight="bold",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="#DBEAFE", alpha=0.8))
ax.grid(True, axis="y", alpha=0.25)

# Panel B: Total training time (3 seeds × 50 epochs)
ax = axes[1]
ft_total_min   = FT_EPOCH_TIME_S   * N_EPOCHS * N_SEEDS / 60
adst_total_min = ADST_EPOCH_TIME_S * N_EPOCHS * N_SEEDS / 60
bars2 = ax.bar(["FT-Transformer", "ADST"],
               [ft_total_min, adst_total_min],
               color=[C_FT, C_ADST], width=0.5)
ax.bar_label(bars2,
             labels=[f"{ft_total_min:.0f} min\n({ft_total_min/60:.1f} hrs)",
                     f"{adst_total_min:.0f} min\n({adst_total_min/60:.1f} hrs)"],
             padding=3, fontweight="bold")
ax.set_ylabel("Total Training Time (minutes)")
ax.set_title(f"(b) Total Training Time\n(3 seeds × {N_EPOCHS} epochs)")
ax.set_ylim(0, ft_total_min * 1.3)
ax.grid(True, axis="y", alpha=0.25)

# Panel C: Time-to-accuracy (minutes to reach each accuracy threshold)
ax = axes[2]
thresholds = [80, 82, 84, 85, 86, 86.5, 87]

def epochs_to_reach(histories, threshold):
    times = []
    for h in histories:
        for i, v in enumerate(h["val_acc"].values * 100):
            if v >= threshold:
                times.append(i + 1)
                break
    return np.mean(times) if times else None

colors_thresh = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(thresholds)))
ft_times   = []
adst_times = []
valid_thresh = []

for thr in thresholds:
    ft_t   = epochs_to_reach(ft_hist,   thr)
    adst_t = epochs_to_reach(adst_hist, thr)
    if ft_t and adst_t:
        ft_times.append(ft_t * FT_EPOCH_TIME_S / 60)
        adst_times.append(adst_t * ADST_EPOCH_TIME_S / 60)
        valid_thresh.append(thr)

ax.plot(valid_thresh, ft_times,   "o-", color=C_FT,   lw=2.5, ms=8,
        label="FT-Transformer")
ax.plot(valid_thresh, adst_times, "s-", color=C_ADST, lw=2.5, ms=8,
        label="ADST")

# Fill — ADST advantage region
ax.fill_between(valid_thresh, ft_times, adst_times,
                where=[f > a for f, a in zip(ft_times, adst_times)],
                color=C_ADST, alpha=0.15, label="ADST faster")

ax.set_xlabel("Target Validation Accuracy (%)")
ax.set_ylabel("Wall-clock Time to Reach (minutes)")
ax.set_title("(c) Time to Reach Target Accuracy\n(lower = faster)")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.25)

fig.suptitle("Training Efficiency: ADST vs Flat FT-Transformer",
             fontsize=14, fontweight="bold")
plt.tight_layout()
save(fig, "04_training_time_comparison")


# ══════════════════════════════════════════════════════════
# PLOT 05 — Seed Stability Boxplot
# ══════════════════════════════════════════════════════════
print("Plot 05: Seed stability...")

fig, ax = plt.subplots(figsize=(8, 6))

ft_vals   = [r["test_acc"]*100 for r in ft_sum]   if ft_sum   else []
adst_vals = [r["test_acc"]*100 for r in adst_sum] if adst_sum else []

bp = ax.boxplot([ft_vals, adst_vals],
                labels=["Flat FT-Transformer", "ADST\n(Semantic Tokens)"],
                patch_artist=True,
                widths=0.4,
                medianprops=dict(color="white", lw=2.5),
                whiskerprops=dict(lw=1.5),
                capprops=dict(lw=1.5),
                flierprops=dict(marker="o", ms=8))

bp["boxes"][0].set_facecolor(C_FT)
bp["boxes"][1].set_facecolor(C_ADST)

# Plot individual seed points
for i, (vals, color) in enumerate([(ft_vals, C_FT), (adst_vals, C_ADST)], 1):
    jitter = np.random.uniform(-0.04, 0.04, len(vals))
    ax.scatter([i + j for j in jitter], vals, color="white",
               edgecolors=color, s=80, zorder=5, lw=2)
    for j, v in enumerate(vals):
        seed_label = f"seed={SEEDS[j]}"
        ax.annotate(seed_label, (i + jitter[j], v),
                    xytext=(12, 0), textcoords="offset points",
                    fontsize=8, color=color, va="center")

# Stats annotation
for i, vals in enumerate([ft_vals, adst_vals], 1):
    if vals:
        ax.text(i, min(vals) - 0.08,
                f"μ={np.mean(vals):.2f}%\nσ={np.std(vals):.2f}pp",
                ha="center", fontsize=9, color=["gray", C_ADST][i-1])

ax.set_ylabel("Test Accuracy (%)")
ax.set_title("Training Stability Across Seeds\n"
             "Smaller box = more consistent across different initializations")
ax.set_ylim(min(min(ft_vals), min(adst_vals)) - 0.5,
             max(max(ft_vals), max(adst_vals)) + 0.5)
ax.grid(True, axis="y", alpha=0.25)
save(fig, "05_stability_boxplot")


# ══════════════════════════════════════════════════════════
# PLOT 06 — Final Test Accuracy Bar (ADST vs FT vs XGBoost)
# ══════════════════════════════════════════════════════════
print("Plot 06: Test accuracy bar chart...")

fig, ax = plt.subplots(figsize=(9, 6))

models  = ["XGBoost\n(Tree-based)", "Flat FT-Transformer\n(130 tokens)",
           "ADST\n(7 semantic tokens)"]
means   = [XGBOOST_ACC*100,
           np.mean(ft_test_accs)*100   if ft_test_accs   else 0,
           np.mean(adst_test_accs)*100 if adst_test_accs else 0]
stds    = [0,
           np.std(ft_test_accs)*100   if len(ft_test_accs)>1   else 0,
           np.std(adst_test_accs)*100 if len(adst_test_accs)>1 else 0]
colors  = [C_XGB, C_FT, C_ADST]

bars = ax.bar(models, means, yerr=stds, color=colors, width=0.5,
              error_kw=dict(elinewidth=2, capsize=8, capthick=2),
              zorder=3)

for bar, m, s in zip(bars, means, stds):
    label = f"{m:.2f}%"
    if s > 0:
        label += f"\n(±{s:.2f}pp)"
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + s + 0.15,
            label, ha="center", va="bottom",
            fontsize=11, fontweight="bold")

# Annotate param counts
param_labels = ["—", f"139K params", f"140K params"]
for i, (bar, plabel) in enumerate(zip(bars, param_labels)):
    ax.text(bar.get_x() + bar.get_width()/2, means[i] - 0.4,
            plabel, ha="center", va="top", fontsize=9, color="white",
            fontweight="bold")

ax.set_ylabel("Test Accuracy (%)")
ax.set_title("Final Test Accuracy Comparison\n"
             "Error bars = ±1 std across 3 seeds")
ax.set_ylim(84, 92.5)
ax.grid(True, axis="y", alpha=0.25, zorder=0)

# Add interpretability annotation for ADST
ax.annotate("+ Interpretable\ngroup attention",
            xy=(2, means[2]),
            xytext=(2.35, means[2] + 2.5),
            fontsize=9, color=C_ADST, fontweight="bold",
            arrowprops=dict(arrowstyle="->", color=C_ADST, lw=1.5))

save(fig, "06_test_accuracy_bar")


# ══════════════════════════════════════════════════════════
# PLOT 07 — Per-Class Recall Heatmap (24 classes × 3 models)
# ══════════════════════════════════════════════════════════
print("Plot 07: Per-class recall heatmap...")

if ft_sum and adst_sum:
    class_names = list(ft_sum[0]["per_class_recall"].keys())

    ft_rec   = {c: np.mean([r["per_class_recall"][c] for r in ft_sum])
                for c in class_names}
    adst_rec = {c: np.mean([r["per_class_recall"][c] for r in adst_sum])
                for c in class_names}
    xgb_rec  = {c: XGB_RECALLS.get(c, 0) for c in class_names}

    # Sort by XGBoost recall descending
    sorted_cls = sorted(class_names, key=lambda c: -xgb_rec[c])

    matrix = np.array([[xgb_rec[c], ft_rec[c], adst_rec[c]]
                       for c in sorted_cls])

    fig, ax = plt.subplots(figsize=(8, 13))
    cmap = LinearSegmentedColormap.from_list(
        "recall", ["#FEE2E2", "#FEF3C7", "#D1FAE5", "#065F46"], N=256)
    im = ax.imshow(matrix, cmap=cmap, aspect="auto", vmin=0.5, vmax=1.0)

    ax.set_xticks([0, 1, 2])
    ax.set_xticklabels(["XGBoost", "FT-Transformer", "ADST"],
                        fontsize=12, fontweight="bold")
    ax.set_yticks(range(len(sorted_cls)))
    ax.set_yticklabels(sorted_cls, fontsize=9)

    # Cell values
    for i in range(len(sorted_cls)):
        for j, val in enumerate(matrix[i]):
            color = "white" if val > 0.85 else "black"
            ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                    fontsize=8.5, color=color, fontweight="bold")

    plt.colorbar(im, ax=ax, label="Recall", shrink=0.5)
    ax.set_title("Per-Class Recall: XGBoost vs FT-Transformer vs ADST\n"
                 "(sorted by XGBoost recall — green=high, red=low)",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    save(fig, "07_per_class_recall_heatmap")


# ══════════════════════════════════════════════════════════
# PLOT 08 — Delta Plot: Where ADST Beats FT (KEY INSIGHT)
# ══════════════════════════════════════════════════════════
print("Plot 08: ADST vs FT delta plot...")

if ft_sum and adst_sum:
    class_names = list(ft_sum[0]["per_class_recall"].keys())
    ft_rec   = {c: np.mean([r["per_class_recall"][c] for r in ft_sum])
                for c in class_names}
    adst_rec = {c: np.mean([r["per_class_recall"][c] for r in adst_sum])
                for c in class_names}

    deltas = {c: (adst_rec[c] - ft_rec[c]) * 100 for c in class_names}
    sorted_cls = sorted(class_names, key=lambda c: deltas[c])
    delta_vals = [deltas[c] for c in sorted_cls]
    bar_colors = [C_WIN if d > 0 else C_LOSE for d in delta_vals]

    fig, ax = plt.subplots(figsize=(10, 11))
    bars = ax.barh(range(len(sorted_cls)), delta_vals,
                   color=bar_colors, alpha=0.85, height=0.7)

    ax.axvline(0, color="black", lw=1.5)
    ax.set_yticks(range(len(sorted_cls)))
    ax.set_yticklabels(sorted_cls, fontsize=9)
    ax.set_xlabel("Recall Delta: ADST minus FT-Transformer (pp)\n"
                  "Positive = ADST higher recall   |   Negative = FT higher recall")
    ax.set_title("Where ADST Outperforms Flat FT-Transformer\n"
                 "Per-class recall difference (mean across 3 seeds)",
                 fontsize=13, fontweight="bold")

    # Value labels
    for i, (bar, val) in enumerate(zip(bars, delta_vals)):
        ax.text(val + (0.05 if val >= 0 else -0.05),
                bar.get_y() + bar.get_height()/2,
                f"{val:+.2f}pp", va="center",
                ha="left" if val >= 0 else "right",
                fontsize=8.5, fontweight="bold",
                color=C_WIN if val > 0 else C_LOSE)

    adst_wins  = sum(1 for d in delta_vals if d > 0)
    adst_loses = sum(1 for d in delta_vals if d < 0)
    ax.text(0.98, 0.02,
            f"ADST wins:  {adst_wins}/24 classes\n"
            f"FT wins:    {adst_loses}/24 classes",
            transform=ax.transAxes, ha="right", va="bottom",
            fontsize=11, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.4", facecolor="#F0FDF4",
                      edgecolor=C_WIN, lw=1.5))

    # Patch legend
    win_patch  = mpatches.Patch(color=C_WIN,  label="ADST higher recall")
    lose_patch = mpatches.Patch(color=C_LOSE, label="FT-Transformer higher recall")
    ax.legend(handles=[win_patch, lose_patch], loc="upper left", fontsize=10)
    ax.grid(True, axis="x", alpha=0.25)
    plt.tight_layout()
    save(fig, "08_adst_wins_delta")


# ══════════════════════════════════════════════════════════
# PLOT 09 — ADST Semantic Group Attention
# ══════════════════════════════════════════════════════════
print("Plot 09: Group attention...")

if adst_sum and "group_attention" in adst_sum[0]:
    groups = list(adst_sum[0]["group_attention"].keys())
    means  = {g: np.mean([r["group_attention"][g] for r in adst_sum])
               for g in groups}
    stds   = {g: np.std( [r["group_attention"][g] for r in adst_sum])
               for g in groups}

    sorted_g    = sorted(groups, key=lambda g: -means[g])
    vals        = [means[g]*100 for g in sorted_g]
    errs        = [stds[g]*100  for g in sorted_g]
    uniform_pct = 100/len(groups)
    bar_cols    = [C_ADST if means[g]*100 > uniform_pct else "#CBD5E1"
                   for g in sorted_g]

    # Readable group labels with descriptions
    labels_desc = {
        "global_context":    "Global Context\n(file_frag_rate)",
        "temporal_splt":     "Temporal Sequence\n(first 10 packets)",
        "recon_cardinality": "Recon / Cardinality\n(scan patterns + fan-out)",
        "tcp_flags":         "TCP Flag Behavior\n(SYN/ACK/FIN/RST counts)",
        "flow_volume":       "Flow Volume\n(duration, bytes, packets)",
        "gre_header":        "GRE Encapsulation\n(Mirai subtype signals)",
        "fragmentation":     "Fragmentation\n(frag_mf, frag_offset)",
    }
    readable = [labels_desc.get(g, g) for g in sorted_g]

    fig, ax = plt.subplots(figsize=(11, 7))
    bars = ax.bar(range(len(sorted_g)), vals, yerr=errs,
                  color=bar_cols, width=0.6,
                  error_kw=dict(elinewidth=2, capsize=6, capthick=2))

    ax.axhline(uniform_pct, color="gray", lw=2, ls="--",
               label=f"Uniform baseline ({uniform_pct:.1f}% each)")

    for bar, val, err in zip(bars, vals, errs):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + err + 0.5,
                f"{val:.1f}%", ha="center", va="bottom",
                fontsize=10, fontweight="bold")

    ax.set_xticks(range(len(sorted_g)))
    ax.set_xticklabels(readable, fontsize=9.5)
    ax.set_ylabel("CLS Token Attention Weight (%)")
    ax.set_title("ADST: How Much the Model Relies on Each Semantic Group\n"
                 "Error bars = ±1 std across 3 seeds  |  Blue = above uniform",
                 fontsize=13)
    ax.legend(fontsize=10)
    ax.set_ylim(0, max(vals) + max(errs) + 5)
    ax.grid(True, axis="y", alpha=0.25)

    # Insight annotations
    ax.annotate("Strongest signal:\nFile-level frag rate\nidentifies frag attacks",
                xy=(0, vals[0]), xytext=(0.5, vals[0] + max(errs) + 3),
                fontsize=8, color=C_ADST, ha="center",
                arrowprops=dict(arrowstyle="->", color=C_ADST))

    save(fig, "09_group_attention")


# ══════════════════════════════════════════════════════════
# PLOT 10 — Accuracy vs Training Time Scatter
# ══════════════════════════════════════════════════════════
print("Plot 10: Efficiency scatter...")

fig, ax = plt.subplots(figsize=(10, 7))

# Total training time in minutes (single seed × 50 epochs)
ft_time_min    = FT_EPOCH_TIME_S   * N_EPOCHS / 60
adst_time_min  = ADST_EPOCH_TIME_S * N_EPOCHS / 60
xgb_time_min   = 30   # approximate XGBoost training time

model_data = [
    ("XGBoost",             xgb_time_min,  XGBOOST_ACC*100, 0,   C_XGB,  800),
    ("Flat FT-Transformer", ft_time_min,   np.mean(ft_test_accs)*100,
     np.std(ft_test_accs)*100, C_FT, 800),
    ("ADST",                adst_time_min, np.mean(adst_test_accs)*100,
     np.std(adst_test_accs)*100, C_ADST, 800),
]

for name, x, y, yerr, color, size in model_data:
    ax.scatter(x, y, color=color, s=size, zorder=5,
               edgecolors="white", lw=2)
    if yerr > 0:
        ax.errorbar(x, y, yerr=yerr, fmt="none", color=color,
                    elinewidth=2, capsize=6)
    offset = {"XGBoost": (5, 5), "Flat FT-Transformer": (5, -10),
              "ADST": (5, 5)}
    ox, oy = offset.get(name, (5, 5))
    ax.annotate(f"{name}\n{y:.2f}%\n{x:.0f} min",
                xy=(x, y), xytext=(ox, oy),
                textcoords="offset points",
                fontsize=10, color=color, fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.3",
                          facecolor="white", edgecolor=color,
                          alpha=0.9))

# Efficiency frontier
ax.plot([adst_time_min, ft_time_min],
        [np.mean(adst_test_accs)*100, np.mean(ft_test_accs)*100],
        "k--", lw=1, alpha=0.3)

# ADST advantage shading
ax.axvspan(0, adst_time_min + 2, alpha=0.04, color=C_ADST)
ax.text(adst_time_min/2, 85.5, "ADST\nzone", ha="center",
        fontsize=9, color=C_ADST, alpha=0.7)

ax.set_xlabel("Total Training Time — Single Seed × 50 Epochs (minutes)")
ax.set_ylabel("Test Accuracy (%)")
ax.set_title("Accuracy vs Training Time: Efficiency Frontier\n"
             "Upper-left = best (high accuracy, low training cost)",
             fontsize=13)
ax.grid(True, alpha=0.25)
ax.set_xlim(-5, ft_time_min + 30)
ax.set_ylim(85, 91.5)
save(fig, "10_efficiency_scatter")


# ══════════════════════════════════════════════════════════
# PLOT 11 — Combined 3×2 Paper Figure
# ══════════════════════════════════════════════════════════
print("Plot 11: Combined paper figure...")

fig = plt.figure(figsize=(16, 14))
gs  = GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.32)

# (a) Val accuracy curves — top left
ax1 = fig.add_subplot(gs[0, 0])
e_ft, m_ft, s_ft = mean_std(ft_hist,   "val_acc")
e_ad, m_ad, s_ad = mean_std(adst_hist, "val_acc")
ax1.plot(e_ft, m_ft*100, color=C_FT,   lw=2, label="FT-Transformer")
ax1.fill_between(e_ft, (m_ft-s_ft)*100, (m_ft+s_ft)*100, color=C_FT, alpha=0.12)
ax1.plot(e_ad, m_ad*100, color=C_ADST, lw=2, label="ADST")
ax1.fill_between(e_ad, (m_ad-s_ad)*100, (m_ad+s_ad)*100, color=C_ADST, alpha=0.12)
ax1.axhline(XGBOOST_ACC*100, color=C_XGB, lw=1.5, ls="--",
            alpha=0.7, label="XGBoost")
ax1.set_title("(a) Validation Accuracy", fontsize=12)
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Val Accuracy (%)")
ax1.legend(fontsize=8); ax1.grid(True, alpha=0.2)
ax1.set_ylim(74, 92)

# (b) Training time — top right
ax2 = fig.add_subplot(gs[0, 1])
b = ax2.bar(["FT-Transformer\n(130 tokens)", "ADST\n(7 tokens)"],
            [FT_EPOCH_TIME_S, ADST_EPOCH_TIME_S],
            color=[C_FT, C_ADST], width=0.45)
ax2.bar_label(b, labels=[f"{FT_EPOCH_TIME_S}s/epoch",
                           f"{ADST_EPOCH_TIME_S}s/epoch"],
              padding=3, fontweight="bold", fontsize=10)
ax2.text(0.5, 0.88, f"{FT_EPOCH_TIME_S//ADST_EPOCH_TIME_S}× faster",
         transform=ax2.transAxes, ha="center", fontsize=12,
         color=C_ADST, fontweight="bold")
ax2.set_title("(b) Training Time per Epoch", fontsize=12)
ax2.set_ylabel("Seconds"); ax2.set_ylim(0, 135)
ax2.grid(True, axis="y", alpha=0.2)

# (c) Test accuracy bar — middle left
ax3 = fig.add_subplot(gs[1, 0])
_models = ["XGBoost", "FT-Transformer", "ADST"]
_means  = [XGBOOST_ACC*100,
           np.mean(ft_test_accs)*100 if ft_test_accs else 0,
           np.mean(adst_test_accs)*100 if adst_test_accs else 0]
_stds   = [0,
           np.std(ft_test_accs)*100 if len(ft_test_accs)>1 else 0,
           np.std(adst_test_accs)*100 if len(adst_test_accs)>1 else 0]
_colors = [C_XGB, C_FT, C_ADST]
bars3   = ax3.bar(_models, _means, yerr=_stds, color=_colors, width=0.45,
                  error_kw=dict(elinewidth=1.5, capsize=5))
for bar, m, s in zip(bars3, _means, _stds):
    lbl = f"{m:.2f}%" + (f"\n±{s:.2f}pp" if s > 0 else "")
    ax3.text(bar.get_x()+bar.get_width()/2, bar.get_height()+s+0.1,
             lbl, ha="center", va="bottom", fontsize=9, fontweight="bold")
ax3.set_title("(c) Test Accuracy", fontsize=12)
ax3.set_ylabel("Test Accuracy (%)"); ax3.set_ylim(84, 92)
ax3.grid(True, axis="y", alpha=0.2)

# (d) Delta plot (top-7 and bottom-5 classes) — middle right
ax4 = fig.add_subplot(gs[1, 1])
if ft_sum and adst_sum:
    class_names = list(ft_sum[0]["per_class_recall"].keys())
    ft_rec2   = {c: np.mean([r["per_class_recall"][c] for r in ft_sum])
                 for c in class_names}
    adst_rec2 = {c: np.mean([r["per_class_recall"][c] for r in adst_sum])
                 for c in class_names}
    deltas2   = {c: (adst_rec2[c] - ft_rec2[c])*100 for c in class_names}
    # Show only top-6 ADST wins and bottom-6 FT wins
    sorted_d  = sorted(class_names, key=lambda c: deltas2[c])
    show_cls  = sorted_d[:6] + sorted_d[-6:]
    show_vals = [deltas2[c] for c in show_cls]
    show_cols = [C_WIN if d > 0 else C_LOSE for d in show_vals]
    ax4.barh(range(len(show_cls)), show_vals, color=show_cols, alpha=0.85)
    ax4.axvline(0, color="black", lw=1.5)
    ax4.set_yticks(range(len(show_cls)))
    ax4.set_yticklabels([c.replace("DDoS-","").replace("DoS-","")
                         for c in show_cls], fontsize=8)
    ax4.set_xlabel("Recall Δ (ADST − FT, pp)")
    ax4.set_title("(d) Per-Class: ADST vs FT-Transformer", fontsize=12)
    ax4.grid(True, axis="x", alpha=0.2)

# (e) Group attention — bottom left
ax5 = fig.add_subplot(gs[2, 0])
if adst_sum and "group_attention" in adst_sum[0]:
    sorted_g2 = sorted(groups, key=lambda g: -means[g])
    vals5     = [means[g]*100 for g in sorted_g2]
    col5      = [C_ADST if v > uniform_pct else "#CBD5E1" for v in vals5]
    ax5.bar(range(len(sorted_g2)), vals5, color=col5, width=0.6)
    ax5.axhline(uniform_pct, color="gray", lw=1.5, ls="--")
    short_labels = [g.replace("_", "\n") for g in sorted_g2]
    ax5.set_xticks(range(len(sorted_g2)))
    ax5.set_xticklabels(short_labels, fontsize=7.5)
    ax5.set_ylabel("Attention Weight (%)")
    ax5.set_title("(e) ADST Group Attention", fontsize=12)
    ax5.grid(True, axis="y", alpha=0.2)

# (f) Stability boxplot — bottom right
ax6 = fig.add_subplot(gs[2, 1])
if ft_test_accs and adst_test_accs:
    bp6 = ax6.boxplot([ft_test_accs_pct := [v*100 for v in ft_test_accs],
                        adst_test_accs_pct := [v*100 for v in adst_test_accs]],
                       labels=["FT-Transformer", "ADST"],
                       patch_artist=True, widths=0.35,
                       medianprops=dict(color="white", lw=2))
    bp6["boxes"][0].set_facecolor(C_FT)
    bp6["boxes"][1].set_facecolor(C_ADST)
    for i, (vals_pct, c) in enumerate([(ft_test_accs_pct, C_FT),
                                        (adst_test_accs_pct, C_ADST)], 1):
        ax6.scatter([i]*len(vals_pct), vals_pct, color="white",
                    edgecolors=c, s=60, zorder=5, lw=2)
    ax6.set_ylabel("Test Accuracy (%)")
    ax6.set_title("(f) Seed Stability", fontsize=12)
    ax6.grid(True, axis="y", alpha=0.2)

fig.suptitle("IoT IDS: ADST vs Flat FT-Transformer — Complete Comparison\n"
             "CICIoT2023 Dataset — 24 Attack Classes — 3-Seed Evaluation",
             fontsize=14, fontweight="bold", y=1.01)
save(fig, "11_paper_figure_3x2")


# ── Summary ────────────────────────────────────────────────
print()
print("=" * 60)
print("ALL 11 GRAPHS SAVED")
print("=" * 60)
graphs = [
    ("01_val_accuracy_curve.png",       "Val accuracy per epoch (ADST vs FT)"),
    ("02_training_loss_curve.png",      "Training loss per epoch"),
    ("03_convergence_speed.png",        "Who learns faster at key epochs"),
    ("04_training_time_comparison.png", "Time per epoch + total + time-to-accuracy"),
    ("05_stability_boxplot.png",        "Seed-to-seed consistency"),
    ("06_test_accuracy_bar.png",        "Final test acc — all 3 models"),
    ("07_per_class_recall_heatmap.png", "24-class recall heatmap"),
    ("08_adst_wins_delta.png",          "Where ADST beats FT per class"),
    ("09_group_attention.png",          "ADST semantic group attention"),
    ("10_efficiency_scatter.png",       "Accuracy vs training time frontier"),
    ("11_paper_figure_3x2.png",         "Combined 3×2 publication figure"),
]
for fname, desc in graphs:
    path  = os.path.join(OUTPUT_DIR, fname)
    exists = os.path.exists(path)
    print(f"  {'✓' if exists else '✗'}  {fname:<38} {desc}")
print()
print("RECOMMENDED FOR PAPER:")
print("  11_paper_figure_3x2.png      ← main comparison figure")
print("  08_adst_wins_delta.png       ← where ADST wins (key argument)")
print("  09_group_attention.png       ← interpretability contribution")
print("  04_training_time_comparison.png ← efficiency argument")

Loading data...
  FT seeds loaded:   3
  ADST seeds loaded: 3

Plot 01: Val accuracy curves...
  Saved → comparison_plots/01_val_accuracy_curve.png
Plot 02: Training loss curves...
  Saved → comparison_plots/02_training_loss_curve.png
Plot 03: Convergence speed...
  Saved → comparison_plots/03_convergence_speed.png
Plot 04: Training time comparison...
  Saved → comparison_plots/04_training_time_comparison.png
Plot 05: Seed stability...


/tmp/ipykernel_58/1038559812.py:363: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot([ft_vals, adst_vals],


  Saved → comparison_plots/05_stability_boxplot.png
Plot 06: Test accuracy bar chart...
  Saved → comparison_plots/06_test_accuracy_bar.png
Plot 07: Per-class recall heatmap...
  Saved → comparison_plots/07_per_class_recall_heatmap.png
Plot 08: ADST vs FT delta plot...
  Saved → comparison_plots/08_adst_wins_delta.png
Plot 09: Group attention...
  Saved → comparison_plots/09_group_attention.png
Plot 10: Efficiency scatter...
  Saved → comparison_plots/10_efficiency_scatter.png
Plot 11: Combined paper figure...


/tmp/ipykernel_58/1038559812.py:780: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp6 = ax6.boxplot([ft_test_accs_pct := [v*100 for v in ft_test_accs],


  Saved → comparison_plots/11_paper_figure_3x2.png

ALL 11 GRAPHS SAVED
  ✓  01_val_accuracy_curve.png              Val accuracy per epoch (ADST vs FT)
  ✓  02_training_loss_curve.png             Training loss per epoch
  ✓  03_convergence_speed.png               Who learns faster at key epochs
  ✓  04_training_time_comparison.png        Time per epoch + total + time-to-accuracy
  ✓  05_stability_boxplot.png               Seed-to-seed consistency
  ✓  06_test_accuracy_bar.png               Final test acc — all 3 models
  ✓  07_per_class_recall_heatmap.png        24-class recall heatmap
  ✓  08_adst_wins_delta.png                 Where ADST beats FT per class
  ✓  09_group_attention.png                 ADST semantic group attention
  ✓  10_efficiency_scatter.png              Accuracy vs training time frontier
  ✓  11_paper_figure_3x2.png                Combined 3×2 publication figure

RECOMMENDED FOR PAPER:
  11_paper_figure_3x2.png      ← main comparison figure
  08_adst_wins_delta.png